programı çalıştıran kod blogu



In [ ]:
import cv2
import numpy as np
import tensorflow as tf

# DNN yüz tespiti için kullanılan önceden eğitilmiş model dosyaları
modelFile = r"C:\Users\msdte\Downloads\res10_300x300_ssd_iter_140000.caffemodel"
configFile = r"C:\Users\msdte\Downloads\deploy.prototxt"

# OpenCV'nin DNN modülü ile yüz algılama modeli yükleniyor
net = cv2.dnn.readNetFromCaffe(configFile, modelFile)

# Maske tespiti için TensorFlow Keras modeli yükleniyor
# (Kendi eğittiğin modelin doğru dosya adı ve yolu girilmeli)
model = tf.keras.models.load_model("mask_detector_model_1000x1000.h5")
print("Model yüklendi:")
model.summary()  # Modelin katman yapısı ve parametreleri ekrana yazdırılır

# Kamera açılıyor (0: varsayılan webcam)
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Kamera açılamadı.")
    exit()

while True:
    # Kameradan bir kare (frame) okunuyor
    ret, frame = cap.read()
    if not ret:
        print("Kamera görüntüsü alınamadı.")
        break

    h, w = frame.shape[:2]  # Görüntünün yüksekliği ve genişliği

    # Yüz tespiti için DNN modeline uygun blob oluşturuluyor
    # Görüntü 300x300'e yeniden boyutlandırılır ve normalize edilir
    blob = cv2.dnn.blobFromImage(cv2.resize(frame, (300, 300)), 1.0,
                                 (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()  # Yüz tespiti yapılıyor

    # Algılanan tüm yüzler üzerinde döngü
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]  # Güven skoru alınır
        if confidence > 0.3:  # Güven skoru belli bir eşikten büyükse
            # Yüzün konumu orijinal görüntüye göre ölçeklendirilir
            box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
            startX, startY, endX, endY = box.astype("int")

            # Kenarların görüntü sınırları içinde olduğundan emin ol
            startX = max(0, startX)
            startY = max(0, startY)
            endX = min(w - 1, endX)
            endY = min(h - 1, endY)

            # Yüz bölgesi (ROI) kırpılır
            face = frame[startY:endY, startX:endX]

            try:
                # Yüz görüntüsü model girişine uygun boyuta (224x224) getirilir ve normalize edilir
                face_resized = cv2.resize(face, (224, 224)) / 255.0
                face_input = np.expand_dims(face_resized, axis=0)  # Batch boyutu eklenir

                # Maske var/yok tahmini yapılır
                prediction = model.predict(face_input, verbose=0)[0][0]
                label = "Maske Var" if prediction >= 0.8 else "Maske Yok"
                color = (0, 255, 0) if label == "Maske Var" else (0, 0, 255)  # Yeşil veya kırmızı renk

                # Yüz etrafına renkli kutu çizilir
                cv2.rectangle(frame, (startX, startY), (endX, endY), color, 2)

                # Maske durumu ve olasılık metin olarak ekrana yazılır
                cv2.putText(frame, f"{label} ({prediction:.2f})",
                            (startX, startY - 10), cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, color, 2)

            except Exception as e:
                # Yüz işleme sırasında hata olursa konsola yazılır
                print(f"Yüz işleme hatası: {e}")

    # İşlenmiş görüntü ekranda gösterilir
    cv2.imshow("Maske Tespiti (DNN + CNN)", frame)

    # 'q' tuşuna basılınca döngü ve program sonlandırılır
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Kamera serbest bırakılır ve tüm pencereler kapatılır
cap.release()
cv2.destroyAllWindows()


veri seti eğitimi için kullanılmış kod bloğu.

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

# Veri yükleme
data = []
labels = []

def load_images(folder, label, max_images=1000):
    count = 0
    for img_name in os.listdir(folder):
        if count >= max_images:
            break
        img_path = os.path.join(folder, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (224, 224))
        data.append(img)
        labels.append(label)
        count += 1

# Maksimum 1000'er görsel yükle
load_images(r"C:\Users\msdte\OneDrive\Desktop\69000", 1, max_images=1000)  # Maske var
load_images(r"C:\Users\msdte\OneDrive\Desktop\2", 0, max_images=1000)      # Maske yok

# Normalizasyon
data = np.array(data, dtype="float32") / 255.0
labels = np.array(labels)

# Eğitim/Test ayırımı
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

# Data Augmentation
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

# Model kurulumu
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])

# Early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Model eğitimi
model.fit(datagen.flow(X_train, y_train, batch_size=32),
          validation_data=(X_test, y_test),
          epochs=15,
          callbacks=[early_stop])

# Modeli kaydet
model.save("mask_detector_model_1000x1000.h5")
print("✅ Model başarıyla kaydedildi.")
